# Preprocessing Dataset Resep

Tahap preprocessing untuk sistem chatbot rekomendasi resep berbasis bahan (Skripsi NLP).

Alur di notebook ini:
1. Load dataset `.parquet` (66.419 resep)
2. Sampling 5.000 resep untuk eksperimen awal
3. Cleaning teks (title, steps)
4. Parsing setiap item `ingredients` -> qty, unit, nama bahan bersih
5. Membentuk kolom `ingredient_names` (untuk skor efisiensi bahan) dan `text_for_embedding` (untuk SBERT)
6. Simpan hasil ke parquet, dan ke CSV dengan format yang aman (tidak merusak struktur list)

In [1]:
import pandas as pd
import numpy as np
import re
import ast
import html

pd.set_option("display.max_colwidth", 200)

## 1. Load Dataset

In [2]:
# Ganti path sesuai lokasi file kamu
DATA_PATH = "dataset_resep.parquet"

df = pd.read_parquet(DATA_PATH)
print(df.shape)
df.info()
df.head()

(66419, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66419 entries, 0 to 66418
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   ingredients      66419 non-null  object
 1   title            66419 non-null  object
 2   steps            66419 non-null  object
 3   num_ingredients  66419 non-null  int64 
 4   num_steps        66419 non-null  int64 
 5   char_count       66419 non-null  int64 
dtypes: int64(3), object(3)
memory usage: 3.0+ MB


,ingredients,title,steps,num_ingredients,num_steps,char_count
0,"[1 ekor ikan bandeng, 4 buah tahu, 1 buah tomat, 1 buah wortel, 1 sdm kecap manis, Soun (saya tidak pakai), 5 siung bawang merah, 4 siung bawang putih, 2 butir kemiri, 1/2 sdt merica bubuk, 1/2 sd...",Semur Ikan Bandeng khas Surabaya (kids friendly),"[Goreng tahu dan ikan bandeng. (Utk ikan bandeng saya lumuri dulu dgn bumbu ikan goreng), Tumis bumbu halus hingga harum, lalu masukkan tahu dan ikan bandeng. Tambahkan 2 gelas munjung air. Masak ...",13,3,605
1,"[250 gr udang, Tepung roti, Tepung kobe/tepung serbaguna, Merica, garam]",Udang Tepung Roti,"[Bersihkan udang, copot kepala dan kulitnya. Lumuri dengan garam dan merica. Diamkan beberapa saat, Beri air ke tepung kobe, jangan terlalu encer. Lalu masukan udang kedalamnya, Selanjutnya gulirk...",4,3,297
2,"[10 buah tahu pong, 2 lembar daun jeruk, 3 sdm kecap manis, Secukupnya garam, gula, penyedap rasa, Secukupnya air, Minyak goreng untuk menumis, Bumbu halus :, 1/2 sdt merica, 2 siung bawang putih,...",Tahu kecap,"[Tumis bumbu halus hingga harum, masukkan tahu pong dan beri air sedikit, Tambahkan kecap, daun jeruk, garam, gula dan penyedap rasa sambil koreksi rasa, Masak hingga bumbu meresap]",10,3,373
3,"[2 ons ikan asin gabus, 5 buah cabe merah keriting, 10-15 cabe rawit. merah (sesuaikan), 2 siung bawang merah, 3 siung bawang putih, 2 sdm saos sambal, secukupnya garam,gula pasir, secukupnya air,...",Ikan Asin gabus balado,"[Potong2 ikan asin dan cuci bersih lalu. tiriskan. dan goreng, Setelah matang angkat. tiriskan, Haluskan. cabe bawang. dan panaskan minyak lalu tumis cabe (boleh juga pakai bumbu balado jadi) tumi...",10,4,556
4,"[500 gr kakap merah (potong fillet), 1 butir telur, 1 bungkus Kobe Tepung Bumbu Putih (210gr), secukupnya minyak goreng, 1 buah bawang bombay, 200 ml air, 5 sendok makan saus tomat, secukupnya gul...",Resep Masakan Ikan Kakap Asam Manis,"[Cuci bersih ikan fillet, tiriskan., Kocok lepas 1 butir telur., Masukkan ikan fillet kakap ke dalam kocokan telur dan balur dengan Kobe Tepung Bumbu Putih hingga semua bagian tertutup., Goreng di...",14,10,902


In [3]:
def ensure_list(x):
    """Pastikan kolom ingredients/steps berupa list Python,
    walaupun aslinya tersimpan sebagai string representasi list atau numpy.ndarray."""
    if isinstance(x, (list, np.ndarray)):
        return list(x)
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
        except (ValueError, SyntaxError):
            pass
        return [x]
    return []


df["ingredients"] = df["ingredients"].apply(ensure_list)
df["steps"] = df["steps"].apply(ensure_list)

## 2. Cleaning Awal & Filtering

Buang resep yang bahan/langkahnya kosong, judul kosong, atau duplikat.

In [4]:
import re


def strip_emoji(text):
    emoji_pattern = re.compile(
        "["
        "\U0001f300-\U0001faff"
        "\U00002600-\U000027bf"
        "\U0001f000-\U0001f02f"
        "\ufe0f"  # variation selector yang sering nempel di emoji (mis. ✨️)
        "]+",
        flags=re.UNICODE,
    )
    cleaned = emoji_pattern.sub("", text)
    return re.sub(r"\s+", " ", cleaned).strip()  # rapikan spasi ganda sisa emoji


# terapkan ke kolom title sebelum langkah pembersihan judul lainnya
df["title"] = df["title"].apply(strip_emoji)

In [5]:
before = len(df)

df = df[df["title"].notna() & (df["title"].str.strip() != "")]
df = df[df["ingredients"].apply(lambda x: len(x) > 0)]
df = df[df["steps"].apply(lambda x: len(x) > 0)]

# duplikat berdasarkan judul + bahan
df["_dedup_key"] = (
    df["title"].str.lower().str.strip()
    + "||"
    + df["ingredients"].apply(lambda x: "|".join(x))
)
df = df.drop_duplicates(subset="_dedup_key").drop(columns="_dedup_key")

df = df.reset_index(drop=True)
print(f"Sebelum: {before} resep -> Sesudah cleaning: {len(df)} resep")

Sebelum: 66419 resep -> Sesudah cleaning: 66419 resep


## 3. Sampling 5.000 Resep

Sampling acak dengan random_state tetap supaya reproducible.

In [6]:
N_SAMPLE = 5000
RANDOM_STATE = 42

df_sample = df.sample(n=min(N_SAMPLE, len(df)), random_state=RANDOM_STATE).reset_index(
    drop=True
)
print(df_sample.shape)
df_sample.head()

(5000, 6)


,ingredients,title,steps,num_ingredients,num_steps,char_count
0,"[2 bks mie instan, 1 buah wortel, potong dadu kecil, 5 buah buncis, rajang kecil, 3 butir telur ayam, 100 gr kornet kambing, 2 batang seledri, iris halus, 1 batang daun bawang, iris, 15 lembar kul...",Martabak Mie,"[Rebus wortel hingga setengah matang, kemudian masukkan mie instan. Masak hingga mie setengah matang kemudian tambahkan buncis. Masak hingga semua matang. Matikan api, tiriskan, Masukkan daun bawa...",8,3,634
1,"[2-3 butir telur, 1 lembar daun jeruk, 3 siung bawang merah, iris, 2 siung bawang putih, iris, 1 buah cabai keriting merah, iris, 2 buah cabai rawit domba, iris. Boleh ditambah lagi kalau mau lebi...",Telur Bawang Kecap - Bekal Suami,"[Panaskan wajan, masukan minyak. Lalu tumis bawang merah, bawang putih, cabai keriting, cabai rawit, dan batang bawang daun sampai harum, lalu saya sisihkan., Masih pakai minyak yang sama, ceplok ...",15,2,712
2,"[4 butir telur ayam, 3 batang lombok merah keriting, 3 batang lombok hijau keriting, 1 batang daun bawang, 3 butir bawang putih, 1/2 sdt wijen sangrai, 1 sdm minyak wijen, 50 ml kecap asin, 50 ml ...",166 : Korean Mayak Eggs,"[Cuci bersih bahan marinasi kemudian potong sesuai selera., Rebus air sampai mendidih kemudian rebus telur sampai matang, kupas telur, sisihkan., Masukkan bahan marinasi dalam wadah, aduk hingga t...",10,5,596
3,"[4 potong ayam, 1 ruas jahe, 3 biji bawang putih, 1 batang daun bawang, 1 sdm saos tiram, 1 sdm kecap asin (skip), 1/2 sdm maizena, Secukupnya kaldu bubuk &amp; garam]",Ayam Kukus Jahe,"[Siapkan semua bahan, cuci bersih ayam, cincang jahe & bawang putih, potong2 daun bawang., Siapkan panci kukusan, masukkan ayam ke dalam piring, jahe, bawang putih, saos tiram, kaldu bubuk, garam ...",8,3,463
4,"[300 gr ikan tenggiri, 150 gr tepung tapioka, 2 siung bawang putih goreng, 1 sdt kaldu jamur, 1/2 sdt garam, 1/2 sdt merica, 1/2 sdt gula, 1 butir telur, Secukupnya es batu, Secukupnya kulit pangs...",Baso Tahu Siomay tenggiri,"[Pilet ikan sisihkan, kemudian chopper dengan bawang putih dan air es sampai halus, Tambahkan telur, gula, garam, kaldu bubuk dan merica aduk sebentar., Tambahkan tepung tapioka aduk hingga rata.,...",12,4,490


## 4. Parser Bahan (Ingredient Parser)

Menguraikan tiap item bahan mentah (mis. `"2 butir telur, dikocok lepas"`) menjadi:
- `qty` -> kuantitas (mis. "2")
- `unit` -> satuan, dinormalisasi ke bentuk baku (mis. "btr" -> "butir")
- `name` -> nama bahan bersih (mis. "telur")
- `note` -> catatan/instruksi tambahan (mis. "dikocok lepas")

**Catatan penting soal regex (hasil debugging panjang):**
- `\\b` (word boundary) sengaja HANYA diletakkan SETELAH unit, bukan sebelum -- supaya kuantitas+satuan yang nempel tanpa spasi (mis. `"100gr"`) tetap terbaca. Kalau `\\b` juga dipasang di depan unit, `"gr"` pada `"100gr"` gagal dikenali karena tidak ada transisi word-boundary antara digit dan huruf yang menempel langsung.
- Pola kuantitas mengenali pecahan campuran (`"1 1/2"`), rentang (`"1-2"`), desimal, bilangan bulat, dan kata kuantitas (`"secukupnya"`, `"sesuai selera"`, dst).
- `html.unescape()` dipanggil di awal untuk membereskan entity HTML yang belum ter-decode (mis. `&amp;`).
- Bullet/dash di awal teks (mis. `"-3 butir telur"`) dibuang sebelum parsing.

In [7]:
# Daftar satuan umum dalam resep masakan Indonesia (termasuk singkatan umum)
UNITS = [
    "sendok makan",
    "sendok teh",
    "sdm",
    "sdt",
    "gram",
    "gr",
    "kg",
    "ons",
    "ml",
    "mili",
    "liter",
    "ltr",
    "l",
    "butir",
    "btr",
    "buah",
    "bh",
    "siung",
    "lembar",
    "lbr",
    "batang",
    "btg",
    "ikat",
    "ruas",
    "cm",
    "potong",
    "ptg",
    "biji",
    "bj",
    "ekor",
    "ekr",
    "genggam",
    "bungkus",
    "bks",
    "sachet",
    "sct",
    "papan",
    "buku",
    "helai",
    "hlai",
    "gelas",
    "gls",
    "cup",
    "mangkuk",
    "piring",
    "sisir",
    "kotak",
    "ktk",
    "kaleng",
    "klg",
    "botol",
    "btl",
]
# alias satuan -> bentuk baku (dipakai untuk menyeragamkan field 'unit')
UNIT_ALIASES = {
    "btr": "butir",
    "bh": "buah",
    "sdm": "sendok makan",
    "sdt": "sendok teh",
    "gr": "gram",
    "ltr": "liter",
    "l": "liter",
    "lbr": "lembar",
    "btg": "batang",
    "bks": "bungkus",
    "ptg": "potong",
    "bj": "biji",
    "ekr": "ekor",
    "gls": "gelas",
    "ktk": "kotak",
    "klg": "kaleng",
    "btl": "botol",
    "sct": "sachet",
    "hlai": "helai",
    "mili": "ml",
    "ml": "ml",
}

# urutkan dari yang paling panjang agar regex match unit multi-kata lebih dulu
UNITS_SORTED = sorted(UNITS, key=len, reverse=True)
UNIT_PATTERN = "|".join(re.escape(u) for u in UNITS_SORTED)

# urutan penting: range & pecahan campuran harus dicoba SEBELUM integer biasa
QTY_PATTERN = (
    r"\d+\s*-\s*\d+"  # rentang: 1-2
    r"|\d+\s+\d+\s*/\s*\d+"  # pecahan campuran: 1 1/2
    r"|\d+\s*/\s*\d+"  # pecahan: 1/2
    r"|\d+[.,]\d+"  # desimal: 1.5
    r"|\d+"  # integer biasa
    r"|sesuai selera"  # kata kuantitas
    r"|secukupnya"
    r"|sejumput"
    r"|beberapa"
)

# catatan: \b sengaja HANYA di belakang unit, bukan di depan -- supaya
# qty+unit yang nempel tanpa spasi (mis. '100gr') tetap kebaca, tapi unit
# tidak match sebagian kata di tengah nama bahan (mis. 'l' di 'lengkuas')
INGREDIENT_REGEX = re.compile(
    rf"^\s*(?P<qty>{QTY_PATTERN})?\s*(?:(?P<unit>{UNIT_PATTERN})\b)?\.?\s*(?P<name>.+?)\s*$",
    re.IGNORECASE,
)


def clean_name(name: str) -> str:
    name = name.lower().strip()
    name = re.sub(r"\(.*?\)", "", name)  # buang isi dalam kurung, mis. (opsional)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def parse_ingredient(raw: str) -> dict:
    raw = html.unescape(raw)  # decode &amp; , &quot; , dst.
    raw = raw.strip()
    raw = re.sub(r"^[\-\*•]+\s*", "", raw)  # buang bullet/dash di depan
    if "," in raw:
        main, note = raw.split(",", 1)
    else:
        main, note = raw, ""
    main = main.strip()

    m = INGREDIENT_REGEX.match(main)
    if m and m.group("name"):
        qty = m.group("qty")
        unit = m.group("unit")
        name = clean_name(m.group("name"))
    else:
        qty, unit, name = None, None, clean_name(main)

    unit_clean = unit.strip().lower() if unit else None
    unit_clean = UNIT_ALIASES.get(unit_clean, unit_clean)

    return {
        "raw": raw,
        "qty": qty.strip() if qty else None,
        "unit": unit_clean,
        "name": name,
        "note": note.strip().lower() if note else None,
    }


# quick test -- termasuk semua kasus khusus yang pernah ditemukan selama debugging
for t in [
    "2 butir telur, dikocok lepas",
    "100gr dada ayam",
    "2 siung bawang putih",
    "garam secukupnya",
    "Tepung roti",
    "1/2 sdt merica bubuk",
    "2 bh cabai merah",
    "3 btr telur ayam",
    "-3 butir telur",
    "1 1/2 sdt kaldu ayam bubuk",
    "1-2 sdm kecap manis",
    "cabe sesuai selera",
    "kaldu bubuk &amp; garam",
    "100 mili air",
]:
    print(f"{t!r:40s} -> {parse_ingredient(t)}")

'2 butir telur, dikocok lepas'           -> {'raw': '2 butir telur, dikocok lepas', 'qty': '2', 'unit': 'butir', 'name': 'telur', 'note': 'dikocok lepas'}
'100gr dada ayam'                        -> {'raw': '100gr dada ayam', 'qty': '100', 'unit': 'gram', 'name': 'dada ayam', 'note': None}
'2 siung bawang putih'                   -> {'raw': '2 siung bawang putih', 'qty': '2', 'unit': 'siung', 'name': 'bawang putih', 'note': None}
'garam secukupnya'                       -> {'raw': 'garam secukupnya', 'qty': None, 'unit': None, 'name': 'garam secukupnya', 'note': None}
'Tepung roti'                            -> {'raw': 'Tepung roti', 'qty': None, 'unit': None, 'name': 'tepung roti', 'note': None}
'1/2 sdt merica bubuk'                   -> {'raw': '1/2 sdt merica bubuk', 'qty': '1/2', 'unit': 'sendok teh', 'name': 'merica bubuk', 'note': None}
'2 bh cabai merah'                       -> {'raw': '2 bh cabai merah', 'qty': '2', 'unit': 'buah', 'name': 'cabai merah', 'note': None}
'3 btr 

In [8]:
df_sample["ingredients_parsed"] = df_sample["ingredients"].apply(
    lambda lst: [parse_ingredient(item) for item in lst]
)

# kolom khusus daftar nama bahan bersih -> dipakai utk skor efisiensi bahan nanti
df_sample["ingredient_names"] = df_sample["ingredients_parsed"].apply(
    lambda parsed: [p["name"] for p in parsed if p["name"]]
)

df_sample[["title", "ingredients", "ingredient_names"]].head()

,title,ingredients,ingredient_names
0,Martabak Mie,"[2 bks mie instan, 1 buah wortel, potong dadu kecil, 5 buah buncis, rajang kecil, 3 butir telur ayam, 100 gr kornet kambing, 2 batang seledri, iris halus, 1 batang daun bawang, iris, 15 lembar kul...","[mie instan, wortel, buncis, telur ayam, kornet kambing, seledri, daun bawang, kulit lumpia]"
1,Telur Bawang Kecap - Bekal Suami,"[2-3 butir telur, 1 lembar daun jeruk, 3 siung bawang merah, iris, 2 siung bawang putih, iris, 1 buah cabai keriting merah, iris, 2 buah cabai rawit domba, iris. Boleh ditambah lagi kalau mau lebi...","[telur, daun jeruk, bawang merah, bawang putih, cabai keriting merah, cabai rawit domba, bawang daun, minyak, kecap manis, saos tiram, saos sambal, minyak wijen, kecap asin, ladaku, air]"
2,166 : Korean Mayak Eggs,"[4 butir telur ayam, 3 batang lombok merah keriting, 3 batang lombok hijau keriting, 1 batang daun bawang, 3 butir bawang putih, 1/2 sdt wijen sangrai, 1 sdm minyak wijen, 50 ml kecap asin, 50 ml ...","[telur ayam, lombok merah keriting, lombok hijau keriting, daun bawang, bawang putih, wijen sangrai, minyak wijen, kecap asin, madu, air]"
3,Ayam Kukus Jahe,"[4 potong ayam, 1 ruas jahe, 3 biji bawang putih, 1 batang daun bawang, 1 sdm saos tiram, 1 sdm kecap asin (skip), 1/2 sdm maizena, Secukupnya kaldu bubuk &amp; garam]","[ayam, jahe, bawang putih, daun bawang, saos tiram, kecap asin, maizena, kaldu bubuk & garam]"
4,Baso Tahu Siomay tenggiri,"[300 gr ikan tenggiri, 150 gr tepung tapioka, 2 siung bawang putih goreng, 1 sdt kaldu jamur, 1/2 sdt garam, 1/2 sdt merica, 1/2 sdt gula, 1 butir telur, Secukupnya es batu, Secukupnya kulit pangs...","[ikan tenggiri, tepung tapioka, bawang putih goreng, kaldu jamur, garam, merica, gula, telur, es batu, kulit pangsit, tahu putih, bumbu kacang]"


## 5. Cleaning Teks: Title & Steps

In [9]:
def clean_text(text: str) -> str:
    text = str(text)
    text = html.unescape(text)  # decode HTML entity spt &amp;
    text = re.sub(r"\s+", " ", text)  # rapikan whitespace berlebih
    text = text.strip()
    return text


df_sample["title_clean"] = df_sample["title"].apply(clean_text)
df_sample["steps_clean"] = df_sample["steps"].apply(
    lambda lst: [clean_text(s) for s in lst]
)

# gabungkan steps jadi satu teks (berguna utk ditampilkan ke user nanti)
df_sample["steps_text"] = df_sample["steps_clean"].apply(
    lambda lst: " ".join(f"{i + 1}. {s}" for i, s in enumerate(lst))
)

## 6. Menyusun Teks untuk Embedding SBERT

`text_for_embedding` menggabungkan judul + nama-nama bahan bersih (tanpa qty/unit/note).

> Catatan: model `LazarusNLP/all-indo-e5-small-v4` adalah model keluarga E5, yang butuh prefix `"query: "` atau `"passage: "` saat inference. Prefix ini ditambahkan pada tahap embedding (notebook 02), bukan di sini.

In [10]:
df_sample["text_for_embedding"] = (
    df_sample["title_clean"].str.lower()
    + " - bahan: "
    + df_sample["ingredient_names"].apply(lambda names: ", ".join(names))
)

df_sample[["title_clean", "ingredient_names", "text_for_embedding"]].head(3)

,title_clean,ingredient_names,text_for_embedding
0,Martabak Mie,"[mie instan, wortel, buncis, telur ayam, kornet kambing, seledri, daun bawang, kulit lumpia]","martabak mie - bahan: mie instan, wortel, buncis, telur ayam, kornet kambing, seledri, daun bawang, kulit lumpia"
1,Telur Bawang Kecap - Bekal Suami,"[telur, daun jeruk, bawang merah, bawang putih, cabai keriting merah, cabai rawit domba, bawang daun, minyak, kecap manis, saos tiram, saos sambal, minyak wijen, kecap asin, ladaku, air]","telur bawang kecap - bekal suami - bahan: telur, daun jeruk, bawang merah, bawang putih, cabai keriting merah, cabai rawit domba, bawang daun, minyak, kecap manis, saos tiram, saos sambal, minyak ..."
2,166 : Korean Mayak Eggs,"[telur ayam, lombok merah keriting, lombok hijau keriting, daun bawang, bawang putih, wijen sangrai, minyak wijen, kecap asin, madu, air]","166 : korean mayak eggs - bahan: telur ayam, lombok merah keriting, lombok hijau keriting, daun bawang, bawang putih, wijen sangrai, minyak wijen, kecap asin, madu, air"


## 7. Rekap Statistik & Sanity Check

In [11]:
df_sample["num_ingredients_clean"] = df_sample["ingredient_names"].apply(len)

print("Jumlah resep:", len(df_sample))
print("Missing values:\n", df_sample.isna().sum())
print("\nStatistik jumlah bahan (setelah cleaning):")
print(df_sample["num_ingredients_clean"].describe())
print("\nStatistik jumlah langkah:")
print(df_sample["steps_clean"].apply(len).describe())

Jumlah resep: 5000
Missing values:
 ingredients              0
title                    0
steps                    0
num_ingredients          0
num_steps                0
char_count               0
ingredients_parsed       0
ingredient_names         0
title_clean              0
steps_clean              0
steps_text               0
text_for_embedding       0
num_ingredients_clean    0
dtype: int64

Statistik jumlah bahan (setelah cleaning):
count    5000.000000
mean       12.759000
std         5.268669
min         2.000000
25%         9.000000
50%        12.000000
75%        16.000000
max        48.000000
Name: num_ingredients_clean, dtype: float64

Statistik jumlah langkah:
count    5000.000000
mean        5.145400
std         2.012826
min         2.000000
25%         4.000000
50%         5.000000
75%         6.000000
max        19.000000
Name: steps_clean, dtype: float64


## 8. Simpan Hasil Preprocessing (Parquet)

In [12]:
OUTPUT_PATH = "resep_5000_clean.parquet"

cols_to_save = [
    "title",
    "title_clean",
    "ingredients",
    "ingredients_parsed",
    "ingredient_names",
    "steps",
    "steps_clean",
    "steps_text",
    "text_for_embedding",
    "num_ingredients_clean",
    "num_steps",
    "char_count",
]

df_sample[cols_to_save].to_parquet(OUTPUT_PATH, index=False)
print(f"Tersimpan: {OUTPUT_PATH}")
df_sample[cols_to_save].sample(3)

Tersimpan: resep_5000_clean.parquet


,title,title_clean,ingredients,ingredients_parsed,ingredient_names,steps,steps_clean,steps_text,text_for_embedding,num_ingredients_clean,num_steps,char_count
3835,Homemade Chicken Luncheon,Homemade Chicken Luncheon,"[500 gr paha ayam fillet, 60 gr putih telur, 50 gr bawang putih, haluskan, 85 gr bawang bombay, haluskan, 1,5 sdt angkak, skip, 1/2 sdt merica bubuk, 3/4 sdt garam, 1 sdm kecap asin, 1 sdm kecap i...","[{'raw': '500 gr paha ayam fillet', 'qty': '500', 'unit': 'gram', 'name': 'paha ayam fillet', 'note': None}, {'raw': '60 gr putih telur', 'qty': '60', 'unit': 'gram', 'name': 'putih telur', 'note'...","[paha ayam fillet, putih telur, bawang putih, bawang bombay, 1, merica bubuk, garam, kecap asin, kecap ikan, kaldu bubuk, gula rose brand, minyak goreng rose brand, tepung maizena rose brand, ment...","[Siapkan bahan-bahan yang akan digunakan. Saya potong-potong dadu dada ayamnya agar lebih mudah saat dichopper, Masukkan potongan dada ayam ke dalam chopper, masukkan semua bahan kecuali maizena d...","[Siapkan bahan-bahan yang akan digunakan. Saya potong-potong dadu dada ayamnya agar lebih mudah saat dichopper, Masukkan potongan dada ayam ke dalam chopper, masukkan semua bahan kecuali maizena d...","1. Siapkan bahan-bahan yang akan digunakan. Saya potong-potong dadu dada ayamnya agar lebih mudah saat dichopper 2. Masukkan potongan dada ayam ke dalam chopper, masukkan semua bahan kecuali maize...","homemade chicken luncheon - bahan: paha ayam fillet, putih telur, bawang putih, bawang bombay, 1, merica bubuk, garam, kecap asin, kecap ikan, kaldu bubuk, gula rose brand, minyak goreng rose bran...",15,5,1106
1982,Sop Daging Sapi,Sop Daging Sapi,"[300 gram daging sapi, 1 Liter air, 1 buah wortel, 1 buah kentang, Secukupnya daun bawang, Secukupnya seledri, 3 buah cengkeh, 1 buah kayu manis kecil, 3-4 buah kapulaga, Secukupnya pala dan meric...","[{'raw': '300 gram daging sapi', 'qty': '300', 'unit': 'gram', 'name': 'daging sapi', 'note': None}, {'raw': '1 Liter air', 'qty': '1', 'unit': 'liter', 'name': 'air', 'note': None}, {'raw': '1 bu...","[daging sapi, air, wortel, kentang, daun bawang, seledri, cengkeh, kayu manis kecil, kapulaga, pala dan merica bubuk, garam, bawang putih, bawang merah, sejempol kecil jahe]","[Daging potong kecil ukuran sop. Haluskan bahan halus, bisa dichoper dan tambahkan air atau diuleg halus., Tumis bumbu halus sampe matang dan ga langu, tambahkan merica dan lada bubuk, tambahkan k...","[Daging potong kecil ukuran sop. Haluskan bahan halus, bisa dichoper dan tambahkan air atau diuleg halus., Tumis bumbu halus sampe matang dan ga langu, tambahkan merica dan lada bubuk, tambahkan k...","1. Daging potong kecil ukuran sop. Haluskan bahan halus, bisa dichoper dan tambahkan air atau diuleg halus. 2. Tumis bumbu halus sampe matang dan ga langu, tambahkan merica dan lada bubuk, tambahk...","sop daging sapi - bahan: daging sapi, air, wortel, kentang, daun bawang, seledri, cengkeh, kayu manis kecil, kapulaga, pala dan merica bubuk, garam, bawang putih, bawang merah, sejempol kecil jahe",14,7,983
1195,Tahu Campur,Tahu Campur,"[500 gr Daging sandung lamur / tetelan/rawonan, 30 gr gula merah, 1 ruas lengkuas, 1 barang sereh, 3 lembar daun jeruk, 6 siung bawang merah, 4 siung bawang putih, 1 ruas kunyit, 1/2 sdt jinten, 1...","[{'raw': '500 gr Daging sandung lamur / tetelan/rawonan', 'qty': '500', 'unit': 'gram', 'name': 'daging sandung lamur / tetelan/rawonan', 'note': None}, {'raw': '30 gr gula merah', 'qty': '30', 'u...","[daging sandung lamur / tetelan/rawonan, gula merah, lengkuas, barang sereh, daun jeruk, bawang merah, bawang putih, kunyit, jinten, lada, singkong, bawang putih, jinten bubuk, ketumbar, kunyit, d...","[Tumis bumbu hingga harum lalu masuk ke dalam panci kaldu daging yang sudah di rebus hingga lunak., Siapkan bahan perkedel singkong parut mengunakan parutan kelapa dan haluskan semua bumbu lalu ca...","[Tumis bumbu hingga harum lalu masuk ke dalam panci kaldu daging yang sudah di rebus hingga 

### (Opsional) Ekspor sample ke CSV yang AMAN untuk inspeksi

**Jangan pakai `df.to_csv()` langsung** kalau kolomnya berisi list/dict (`ingredients`, `ingredients_parsed`, `ingredient_names`, `steps`, `steps_clean`) — numpy/pandas bisa menulis elemen list dipisah spasi (bukan koma) kalau tipe datanya sempat berubah jadi `numpy.ndarray` (mis. setelah baca-ulang dari parquet), sehingga saat dibaca ulang pakai `ast.literal_eval`, elemen-elemennya malah nyambung jadi satu string besar dan tidak bisa di-parse balik.

Selalu `json.dumps()` dulu kolom list/dict sebelum diekspor ke CSV.

In [13]:
import json as _json


def to_jsonable(x):
    if hasattr(x, "tolist"):
        x = x.tolist()
    return _json.dumps(x, ensure_ascii=False)


list_like_cols = [
    "ingredients",
    "ingredients_parsed",
    "ingredient_names",
    "steps",
    "steps_clean",
]

df_export = df_sample[cols_to_save].copy()
for col in list_like_cols:
    if col in df_export.columns:
        df_export[col] = df_export[col].apply(to_jsonable)

df_export.to_csv("resep_5000_clean_SAFE.csv", index=False)
print("Tersimpan: resep_5000_clean_SAFE.csv")

# cara baca baliknya:
# import pandas as pd, json
# check = pd.read_csv("resep_5000_clean_SAFE.csv")
# check['ingredient_names'] = check['ingredient_names'].apply(json.loads)

Tersimpan: resep_5000_clean_SAFE.csv
